# STaM Polygon Detection — YOLOv8-seg Training

Binary segmentation: detect the Hebrew text region polygon in manuscript photos.

**Before running**: upload `yolo_polygon.zip` to `My Drive/stam_seg/` and unzip it there.
Structure expected on Drive:
```
My Drive/stam_seg/yolo_polygon/
  data.yaml
  images/{train,val,test}/
  labels/{train,val,test}/
```

In [ ]:
!pip install -q ultralytics

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_YAML   = '/content/drive/MyDrive/stam_seg/yolo_polygon/data.yaml'
RUNS_DIR    = '/content/drive/MyDrive/stam_seg/runs'
RUN_NAME    = 'stam_polygon'
BEST_WEIGHTS = f'{RUNS_DIR}/{RUN_NAME}/weights/best.pt'

# Patch data.yaml path to absolute Drive path so ultralytics finds images
import re, pathlib
yaml_path = pathlib.Path(DATA_YAML)
text = yaml_path.read_text()
text = re.sub(r'^path:.*', f'path: /content/drive/MyDrive/stam_seg/yolo_polygon', text, flags=re.MULTILINE)
yaml_path.write_text(text)
print('data.yaml path updated.')
print(yaml_path.read_text())

In [ ]:
!yolo segment train \
    data={DATA_YAML} \
    model=yolov8n-seg.pt \
    epochs=100 \
    imgsz=640 \
    batch=8 \
    project={RUNS_DIR} \
    name={RUN_NAME} \
    exist_ok=True \
    patience=20

In [ ]:
!yolo segment val \
    model={BEST_WEIGHTS} \
    data={DATA_YAML} \
    imgsz=640

In [ ]:
from ultralytics import YOLO
from IPython.display import Image as IPImage, display
import glob

model = YOLO(BEST_WEIGHTS)
results = model.predict(
    source='/content/drive/MyDrive/stam_seg/yolo_polygon/images/val',
    save=True,
    conf=0.25,
    project='/content/runs',
    name='preview',
    exist_ok=True,
)

# Show first 4 predictions
for p in sorted(glob.glob('/content/runs/preview/*.jpg'))[:4]:
    display(IPImage(p, width=500))

## Results
Training artifacts are saved to `My Drive/stam_seg/runs/stam_polygon/`:
- `weights/best.pt` — best checkpoint by val mAP50
- `weights/last.pt` — last epoch checkpoint
- `results.csv` — per-epoch metrics
- `confusion_matrix.png`, `PR_curve.png` — evaluation plots

Expected: **mAP50 > 0.85** within 50 epochs for this single-class binary polygon task.